# Crypto, Oil & Stocks: Full Colab Project
Run every cell from top to bottom. This notebook creates the SQL database, loads data, demonstrates SQL analytics, and writes and launches the Streamlit dashboard.

Crypto prices are INR; oil prices are USD per barrel and stock values are index points. Oil: Jan 2020–Jan 2026; indices: Jan 2020–Sep 2025; crypto: rolling 365 days. A fixed 2025 crypto comparison requires historical CoinGecko data access beyond the rolling free request.

In [ ]:
%pip -q install pandas requests yfinance streamlit plotly

## 1. Configure API key and date ranges
A CoinGecko Demo key is optional but recommended to reduce rate-limit errors. Add it through the Colab Secrets panel using the name `COINGECKO_API_KEY`.

In [ ]:
import os, time, sqlite3, requests
import pandas as pd
import yfinance as yf
from datetime import datetime

DB_PATH = '/content/market_data.db'
VS_CURRENCY = 'inr'
TOP_COIN_COUNT = 250
HISTORY_DAYS = 365
START_DATE = '2020-01-01'
OIL_END_DATE = '2026-01-01'       # exclusive; includes through 2025-12-31
STOCK_END_DATE = '2025-10-01'    # exclusive; includes September 2025
TICKERS = ['^GSPC', '^IXIC', '^NSEI']
OIL_URL = 'https://raw.githubusercontent.com/datasets/oil-prices/main/data/wti-daily.csv'

try:
    from google.colab import userdata
    COINGECKO_API_KEY = userdata.get('COINGECKO_API_KEY')
except Exception:
    COINGECKO_API_KEY = None
CG_HEADERS = {'x-cg-demo-api-key': COINGECKO_API_KEY} if COINGECKO_API_KEY else {}
print('CoinGecko key configured:', bool(COINGECKO_API_KEY))

## 2. Extract and clean data
CoinGecko markets are paginated in 250-row pages. This collects page 1 (top 250); change `range(1, 2)` to `range(1, 3)` to request a second page too. Historical prices are fetched for the three highest-ranked coins.

In [ ]:
def get_json(url, params=None, headers=None):
    response = requests.get(url, params=params, headers=headers, timeout=60)
    response.raise_for_status()
    return response.json()

# CoinGecko metadata (top market-cap page)
coin_rows = []
for page in range(1, 2):
    coin_rows += get_json('https://api.coingecko.com/api/v3/coins/markets',
        {'vs_currency': VS_CURRENCY, 'order': 'market_cap_desc', 'per_page': 250,
         'page': page, 'sparkline': 'false'}, CG_HEADERS)
coins = pd.DataFrame(coin_rows).sort_values('market_cap_rank').head(TOP_COIN_COUNT)
coin_cols = ['id','symbol','name','current_price','market_cap','market_cap_rank',
             'total_volume','circulating_supply','total_supply','ath','atl','last_updated']
coins = coins[coin_cols].copy()
coins['symbol'] = coins['symbol'].str.upper()
coins['last_updated'] = pd.to_datetime(coins['last_updated'], utc=True).dt.date.astype(str)
coins.rename(columns={'last_updated':'updated_date'}, inplace=True)
top_ids = coins.nsmallest(3, 'market_cap_rank')['id'].tolist()
print('Metadata rows:', len(coins), '| Top 3 selected:', top_ids)

# Daily-ish market chart data for top three coins; endpoint returns a rolling window.
crypto_parts = []
for coin_id in top_ids:
    chart = get_json(f'https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart',
                     {'vs_currency': VS_CURRENCY, 'days': HISTORY_DAYS}, CG_HEADERS)
    part = pd.DataFrame(chart['prices'], columns=['timestamp_ms','price_inr'])
    part['coin_id'] = coin_id
    part['date'] = pd.to_datetime(part['timestamp_ms'], unit='ms', utc=True).dt.date.astype(str)
    crypto_parts.append(part[['coin_id','date','price_inr']])
    time.sleep(1.5)
crypto_prices = pd.concat(crypto_parts, ignore_index=True).drop_duplicates(['coin_id','date'])

# WTI crude oil CSV
oil = pd.read_csv(OIL_URL)
oil_cols = {c.lower(): c for c in oil.columns}
oil.rename(columns={oil_cols['date']:'date', oil_cols['price']:'price_usd'}, inplace=True)
oil['date'] = pd.to_datetime(oil['date'], errors='coerce').dt.strftime('%Y-%m-%d')
oil['price_usd'] = pd.to_numeric(oil['price_usd'], errors='coerce')
oil = oil.dropna(subset=['date'])
oil = oil[(oil.date >= START_DATE) & (oil.date < OIL_END_DATE)][['date','price_usd']]

# Yahoo Finance daily index data
stock_parts = []
for ticker in TICKERS:
    frame = yf.download(ticker, start=START_DATE, end=STOCK_END_DATE, auto_adjust=False, progress=False)
    if frame.empty:
        print('No Yahoo rows returned for', ticker)
        continue
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = frame.columns.get_level_values(0)
    frame = frame.rename_axis('date').reset_index()
    frame['date'] = pd.to_datetime(frame['date']).dt.strftime('%Y-%m-%d')
    frame['ticker'] = ticker
    frame.rename(columns={'Open':'open','High':'high','Low':'low','Close':'close','Volume':'volume'}, inplace=True)
    stock_parts.append(frame[['date','ticker','open','high','low','close','volume']])
stock_prices = pd.concat(stock_parts, ignore_index=True) if stock_parts else pd.DataFrame(
    columns=['date','ticker','open','high','low','close','volume'])
print('Crypto history:', len(crypto_prices), '| Oil rows:', len(oil), '| Stock rows:', len(stock_prices))

## 3. Create SQL tables and load SQLite

In [ ]:
schema = '''
PRAGMA foreign_keys=ON;
CREATE TABLE IF NOT EXISTS cryptocurrencies (
 id TEXT PRIMARY KEY, symbol TEXT NOT NULL, name TEXT NOT NULL, current_price REAL,
 market_cap INTEGER, market_cap_rank INTEGER, total_volume INTEGER,
 circulating_supply REAL, total_supply REAL, ath REAL, atl REAL, updated_date TEXT);
CREATE TABLE IF NOT EXISTS crypto_prices (
 coin_id TEXT NOT NULL REFERENCES cryptocurrencies(id), date TEXT NOT NULL, price_inr REAL NOT NULL,
 PRIMARY KEY(coin_id,date));
CREATE TABLE IF NOT EXISTS oil_prices (date TEXT PRIMARY KEY, price_usd REAL);
CREATE TABLE IF NOT EXISTS stock_prices (
 date TEXT NOT NULL, ticker TEXT NOT NULL, open REAL, high REAL, low REAL, close REAL, volume INTEGER,
 PRIMARY KEY(date,ticker));
CREATE INDEX IF NOT EXISTS idx_crypto_date ON crypto_prices(date);
CREATE INDEX IF NOT EXISTS idx_stock_ticker_date ON stock_prices(ticker,date);
'''
with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(schema)
    conn.execute('DELETE FROM crypto_prices')
    conn.execute('DELETE FROM oil_prices')
    conn.execute('DELETE FROM stock_prices')
    conn.execute('DELETE FROM cryptocurrencies')
    coins.to_sql('cryptocurrencies', conn, if_exists='append', index=False)
    crypto_prices.to_sql('crypto_prices', conn, if_exists='append', index=False)
    oil.to_sql('oil_prices', conn, if_exists='append', index=False)
    stock_prices.to_sql('stock_prices', conn, if_exists='append', index=False)
print('SQLite database saved at', DB_PATH)
with sqlite3.connect(DB_PATH) as conn:
    for table in ['cryptocurrencies','crypto_prices','oil_prices','stock_prices']:
        print(table, pd.read_sql_query(f'SELECT COUNT(*) n FROM {table}', conn).iloc[0,0])

## 4. SQL analytics and cross-market analysis
The examples below use SQLite SQL. Correlation is calculated over daily returns, which is generally more meaningful than correlating raw price levels.

In [ ]:
queries = {
 'Top 3 coins': "SELECT id,symbol,name,market_cap,market_cap_rank FROM cryptocurrencies ORDER BY market_cap_rank LIMIT 3",
 'Supply over 90 percent': "SELECT id,name,circulating_supply,total_supply FROM cryptocurrencies WHERE total_supply>0 AND circulating_supply/total_supply>0.90",
 'Within 10 percent of ATH': "SELECT id,name,current_price,ath FROM cryptocurrencies WHERE ath>0 AND current_price>=ath*0.90",
 'Average rank when volume exceeds 1B INR': "SELECT AVG(market_cap_rank) avg_rank FROM cryptocurrencies WHERE total_volume>1000000000",
 'Yearly average WTI': "SELECT strftime('%Y',date) year,AVG(price_usd) avg_oil FROM oil_prices GROUP BY year ORDER BY year",
 'WTI COVID period': "SELECT * FROM oil_prices WHERE date BETWEEN '2020-03-01' AND '2020-04-30' ORDER BY date",
 'Top S&P daily ranges': "SELECT date,high-low daily_range FROM stock_prices WHERE ticker='^GSPC' ORDER BY daily_range DESC LIMIT 5",
 'Monthly stock average closes': "SELECT ticker,strftime('%Y-%m',date) month,AVG(close) avg_close FROM stock_prices GROUP BY ticker,month ORDER BY month,ticker",
 'Daily cross-market join': "SELECT c.date,c.price_inr bitcoin_inr,o.price_usd oil_usd,s.close sp500,n.close nifty FROM crypto_prices c LEFT JOIN oil_prices o ON o.date=c.date LEFT JOIN stock_prices s ON s.date=c.date AND s.ticker='^GSPC' LEFT JOIN stock_prices n ON n.date=c.date AND n.ticker='^NSEI' WHERE c.coin_id='bitcoin' ORDER BY c.date"
}
with sqlite3.connect(DB_PATH) as conn:
    for label, sql in queries.items():
        print('\n###', label)
        display(pd.read_sql_query(sql, conn).head(10))

# BTC and WTI daily-return correlation on matching dates
returns_sql = """WITH joined AS (
 SELECT c.date,c.price_inr btc,o.price_usd oil,
        LAG(c.price_inr) OVER (ORDER BY c.date) prev_btc,
        LAG(o.price_usd) OVER (ORDER BY c.date) prev_oil
 FROM crypto_prices c JOIN oil_prices o ON o.date=c.date WHERE c.coin_id='bitcoin'
)
SELECT date,btc/prev_btc-1 btc_return,oil/prev_oil-1 oil_return FROM joined
WHERE prev_btc IS NOT NULL AND prev_oil IS NOT NULL"""
with sqlite3.connect(DB_PATH) as conn:
    returns = pd.read_sql_query(returns_sql, conn)
display(returns[['btc_return','oil_return']].corr())

## 5. Write the Streamlit dashboard code
This creates the app file from inside Colab. The SQL runner only executes predefined read-only queries.

In [ ]:
%%writefile /content/app.py
import sqlite3, pandas as pd, plotly.express as px, streamlit as st
DB='/content/market_data.db'
st.set_page_config(page_title='Cross-Market Analysis',page_icon='📈',layout='wide')
st.title('📈 Crypto, Oil & Stocks')
st.caption('SQL-powered market exploration | Crypto in INR, WTI in USD, indices in index points')
@st.cache_data(ttl=120)
def q(sql, params=()):
    with sqlite3.connect(DB) as con: return pd.read_sql_query(sql,con,params=params)
page=st.sidebar.radio('Page',['Market Explorer','Crypto Analysis','SQL Query Runner'])
if page=='Market Explorer':
    bounds=q('SELECT MIN(date) lo,MAX(date) hi FROM crypto_prices')
    lo=pd.to_datetime(bounds.loc[0,'lo']).date(); hi=pd.to_datetime(bounds.loc[0,'hi']).date()
    dates=st.date_input('Date range',(lo,hi),min_value=lo,max_value=hi)
    if len(dates)==2:
        a,b=[d.isoformat() for d in dates]
        metric_sql={
          'Bitcoin avg (INR)':("SELECT AVG(price_inr) v FROM crypto_prices WHERE coin_id='bitcoin' AND date BETWEEN ? AND ?",),
          'WTI avg (USD/barrel)':("SELECT AVG(price_usd) v FROM oil_prices WHERE date BETWEEN ? AND ?",),
          'S&P 500 avg close':("SELECT AVG(close) v FROM stock_prices WHERE ticker='^GSPC' AND date BETWEEN ? AND ?",),
          'NIFTY avg close':("SELECT AVG(close) v FROM stock_prices WHERE ticker='^NSEI' AND date BETWEEN ? AND ?",)}
        cols=st.columns(4)
        for col,(label,(sql,)) in zip(cols,metric_sql.items()):
            v=q(sql,(a,b)).iloc[0,0]; col.metric(label,f'{v:,.2f}' if pd.notna(v) else 'No data')
        snap=q("SELECT c.date,c.price_inr bitcoin_inr,o.price_usd oil_usd,s.close sp500_close,n.close nifty_close FROM crypto_prices c LEFT JOIN oil_prices o ON o.date=c.date LEFT JOIN stock_prices s ON s.date=c.date AND s.ticker='^GSPC' LEFT JOIN stock_prices n ON n.date=c.date AND n.ticker='^NSEI' WHERE c.coin_id='bitcoin' AND c.date BETWEEN ? AND ? ORDER BY c.date",(a,b))
        st.subheader('Daily joined snapshot'); st.dataframe(snap,use_container_width=True,hide_index=True)
        if not snap.empty:
            melted=snap.melt('date',var_name='market',value_name='price')
            st.plotly_chart(px.line(melted,x='date',y='price',color='market'),use_container_width=True)
elif page=='Crypto Analysis':
    coins=q('SELECT DISTINCT coin_id FROM crypto_prices ORDER BY coin_id').coin_id.tolist()
    coin=st.selectbox('Coin',coins,index=coins.index('bitcoin') if 'bitcoin' in coins else 0)
    df=q('SELECT date,price_inr FROM crypto_prices WHERE coin_id=? ORDER BY date',(coin,))
    if not df.empty:
        st.plotly_chart(px.line(df,x='date',y='price_inr',title=f'{coin} daily price (INR)'),use_container_width=True)
        st.dataframe(df,use_container_width=True,hide_index=True)
else:
    sqls={
      'Top 3 by market cap':"SELECT id,symbol,name,market_cap,market_cap_rank FROM cryptocurrencies ORDER BY market_cap_rank LIMIT 3",
      'Supply >90%':"SELECT id,name,circulating_supply,total_supply FROM cryptocurrencies WHERE total_supply>0 AND circulating_supply/total_supply>0.90",
      'Highest BTC daily price':"SELECT MAX(price_inr) highest_inr FROM crypto_prices WHERE coin_id='bitcoin'",
      'Yearly WTI average':"SELECT strftime('%Y',date) year,AVG(price_usd) avg_price FROM oil_prices GROUP BY year ORDER BY year",
      'Monthly stock close':"SELECT ticker,strftime('%Y-%m',date) month,AVG(close) avg_close FROM stock_prices GROUP BY ticker,month ORDER BY month,ticker"}
    label=st.selectbox('Choose SQL query',list(sqls)); st.code(sqls[label],language='sql')
    if st.button('Run query',type='primary'): st.dataframe(q(sqls[label]),use_container_width=True,hide_index=True)

## 6. Launch Streamlit inside Colab
Run this cell after the data load and app creation. Colab's port proxy will display the app in an embedded frame.

In [ ]:
import subprocess, time
from google.colab import output
streamlit_process = subprocess.Popen(['streamlit','run','/content/app.py','--server.port=8501','--server.address=0.0.0.0'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
time.sleep(5)
app_url = output.eval_js('google.colab.kernel.proxyPort(8501)')
print('Streamlit app URL:', app_url)
from IPython.display import IFrame, display
display(IFrame(app_url, width='100%', height=800))

## Project notes
- SQLite database: `/content/market_data.db`; download it from Colab's Files panel if you need the database file.
- To include more metadata coins, request additional CoinGecko pages; historical prices are currently collected only for the top 3.
- Exact-date joins retain missing market days as NULL. The markets have different holidays and trading calendars.
- Correlation is descriptive, not proof that one market causes another to move.
- Yahoo Finance's download wrapper is `yfinance`; its end date is exclusive, so October 1 includes September 2025.